In [ ]:
%python
# ============================================================
# CONFIGURATION — update these two values before running
# ============================================================
catalog = "serverless_stable_ysmqdz_catalog"
schema  = "abac_test"

current_user = spark.sql("SELECT current_user()").collect()[0][0]
print(f"✓ Using: {catalog}.{schema}")
print(f"✓ Running as: {current_user}")


In [ ]:
# ── Cleanup: drop all policies from previous runs to avoid conflicts ──
_all_policies = ['abac_hc_emergency_access', 'abac_hc_regional_access_deny', 'abac_hc_regional_access_east', 'abac_hc_regional_access_north', 'abac_hc_regional_access_south', 'abac_hc_regional_access_west', 'fraud_detection_filter', 'region_row_filter_apac_policy', 'region_row_filter_catalog_apac', 'region_row_filter_catalog_deny', 'region_row_filter_catalog_eu', 'region_row_filter_catalog_us', 'region_row_filter_deny_policy', 'region_row_filter_eu_policy', 'region_row_filter_us_policy', 'flexible_mask_metadata', 'hipaa_mask_contact', 'hipaa_mask_dob', 'hipaa_mask_ids', 'hipaa_mask_name', 'mask_ssn_policy', 'pci_mask_credit_card', 'pci_mask_income', 'pci_mask_ssn', 'pseudo_email_policy', 'redact_other_pii', 'redact_pii', 'clearance_access_policy', 'pii_mask_email', 'pii_mask_name', 'user_access_filter', 'pci_pending_lockdown', 'pci_reviewed_card', 'pci_reviewed_cvv', 'pci_reviewed_name', 'review_complete_classified', 'review_complete_policy', 'review_pending_policy', 'mask_confidential_finance', 'mask_confidential_hr', 'mask_confidential_marketing', 'mask_internal_finance', 'mask_internal_hr', 'mask_internal_marketing', 'telco_billing_confidential', 'telco_billing_restricted', 'telco_cpni_internal', 'telco_cpni_restricted', 'telco_identity_confidential', 'telco_identity_internal', 'telco_identity_restricted', 'telco_service_row_filter_bundle', 'telco_service_row_filter_data', 'telco_service_row_filter_deny', 'telco_service_row_filter_voice']

_dropped, _skipped = 0, 0
for _p in _all_policies:
    try:
        spark.sql(f"DROP POLICY {_p} ON SCHEMA {catalog}.{schema}")
        _dropped += 1
    except Exception:
        _skipped += 1

print(f"Cleanup done — dropped: {_dropped}, not found: {_skipped}")


# Tutorial 4: Secure-by-Default with Control Tags

This tutorial demonstrates two patterns for locking down new data by default and releasing it only after review.

**Part 1 — Manual tagging:** A schema-level control tag (`pending` → `reviewed`) gates access. Admins manually tag PII columns, then flip the control tag to release non-sensitive data.

**Part 2 — Data Classification integration:** Instead of manually tagging PII, Databricks Data Classification automatically detects and tags sensitive columns with `class.*` system governed tags. The control tag pattern ensures no data is visible until a human reviews the auto-classifications.

**Requirements:**

- Databricks Runtime 16.4+ or serverless compute
- A Unity Catalog-enabled workspace
- Governed tags created via the Catalog Explorer UI (see Setup section)
- `CREATE SCHEMA`, `CREATE TABLE`, `CREATE FUNCTION` privileges on the target catalog
- Ownership or `MANAGE` privilege on the schema to create policies
- `APPLY TAG` privilege on schemas, tables, and columns
- `ASSIGN` privilege on the governed tags being used
- `ASSIGN` privilege on `class.*` system governed tags (for Part 2)
- Account group referenced: `abac_tut_admins`
- **Part 2 additionally requires:** Account admin to enable Data Classification on the catalog, and access to `system.data_classification.results`

---
## Part 1: Manual PII Tagging with Control Tags

### Create governed tags

This tutorial uses the following governed tags. Create them in the Catalog Explorer UI if they don't already exist:

| Tag Key | Allowed Values |
|---------|---------------|
| `abac_pii` | `ssn, email, name, phone, address` |
| `abac_review_status` | `pending, reviewed` |

> If these tags already exist from a previous tutorial, skip this step.

In [ ]:
spark.sql(f"""
-- Grant base access so group members can query the tables.
-- ABAC policies handle the fine-grained row/column filtering on top of these grants.
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")


### Create account groups

This tutorial references the following account-level groups: `abac_tut_admins`

Account-level groups (required for Unity Catalog / ABAC) must be created in the **Account Console** (Admin > Groups), not via SQL. `CREATE GROUP` in SQL only creates workspace-local groups which are not compatible with ABAC policies.

> If these groups already exist in your account, skip this step.

In [ ]:
spark.sql(f"""
-- Set the schema-level control tag to 'pending'
-- All new tables will inherit this tag automatically
ALTER SCHEMA {catalog}.{schema}
  SET TAGS ('abac_tut_review_status' = 'pending')
""")


### Step 1: VARIANT Flexible Mask UDF

This UDF detects the runtime data type of a VARIANT value and returns an appropriate masked value. It works across all column types.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.flexible_mask(data VARIANT)
RETURNS VARIANT
RETURN CASE
  WHEN schema_of_variant(data) = 'INT'    THEN 0::VARIANT
  WHEN schema_of_variant(data) = 'BIGINT' THEN -1::VARIANT
  WHEN schema_of_variant(data) = 'DATE'   THEN DATE'1970-01-01'::VARIANT
  WHEN schema_of_variant(data) = 'DOUBLE' THEN 0.00::VARIANT
  WHEN schema_of_variant(data) = 'STRING' THEN '"*******"'::VARIANT
  ELSE NULL::VARIANT
END
""")


### Step 2: Policy 1 — Pending Review (Mask Everything)

When a table has `review_status=pending`, ALL columns are masked. `MATCH COLUMNS TRUE` matches every column. Admins are exempt so they can review the data.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE POLICY review_pending_policy
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.flexible_mask
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
WHEN has_tag_value('abac_tut_review_status', 'pending')
MATCH COLUMNS TRUE AS m ON COLUMN m
""")


### Step 3: Create a Table and Verify Lockdown

The table inherits `review_status=pending` from the schema. All columns are masked for non-admins.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.customer_records")
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.customer_records (
  id INT, full_name STRING, email STRING, ssn STRING,
  department STRING, salary DOUBLE, hire_date DATE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.customer_records VALUES
  (1, 'Alice Johnson',  'alice@acme.com',   '123-45-6789', 'Engineering', 95000.00,  '2020-03-15'),
  (2, 'Bob Smith',      'bob@acme.com',     '234-56-7890', 'Sales',       110000.00, '2019-07-22'),
  (3, 'Carol White',    'carol@acme.com',   '345-67-8901', 'Engineering', 125000.00, '2018-01-10'),
  (4, 'David Lee',      'david@acme.com',   '456-78-9012', 'Marketing',   88000.00,  '2021-05-18'),
  (5, 'Eva Martinez',   'eva@acme.com',     '567-89-0123', 'HR',          102000.00, '2020-11-30')
""")


In [ ]:
spark.sql(f"""
-- As a non-admin: ALL columns are masked
SELECT * FROM {catalog}.{schema}.customer_records
""").show(truncate=False)


### Step 4: Tag PII Columns Manually

An admin reviews the table and tags the sensitive columns.

In [ ]:
spark.sql(f"""
ALTER TABLE {catalog}.{schema}.customer_records
  ALTER COLUMN full_name SET TAGS ('abac_tut_pii' = 'name')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.customer_records
  ALTER COLUMN email SET TAGS ('abac_tut_pii' = 'email')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.customer_records
  ALTER COLUMN ssn SET TAGS ('abac_tut_pii' = 'ssn')
""")


### Step 5: Policy 2 — Reviewed (Mask Only PII)

When a table is marked as `reviewed`, only PII-tagged columns are masked. Non-sensitive columns become visible.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE POLICY review_complete_policy
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.flexible_mask
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
WHEN has_tag_value('abac_tut_review_status', 'reviewed')
MATCH COLUMNS has_tag('abac_tut_pii') AS m
ON COLUMN m
""")


### Step 6: Flip the Control Tag

Mark the table as reviewed. For `customer_records`, the table-level `reviewed` tag overrides the inherited schema-level `pending` tag, so the `pending` policy no longer matches this table. The `reviewed` policy now applies. Other tables in the schema that haven't been flipped still inherit `pending` and remain fully masked.

In [ ]:
spark.sql(f"""
ALTER TABLE {catalog}.{schema}.customer_records
  SET TAGS ('abac_tut_review_status' = 'reviewed')
""")


In [ ]:
spark.sql(f"""
-- id, department, salary, hire_date are now visible
-- full_name, email, ssn remain masked
SELECT * FROM {catalog}.{schema}.customer_records
""").show(truncate=False)


### Step 7: Automatic Inheritance for New Tables

Create another table in the same schema. It inherits `review_status=pending` and is fully locked down automatically.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.transaction_log")
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.transaction_log (
  txn_id INT, account_holder STRING, amount DOUBLE, txn_date DATE, description STRING
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.transaction_log VALUES
  (1001, 'Alice Johnson', 2500.00, '2026-01-15', 'Wire transfer'),
  (1002, 'Bob Smith',     1800.00, '2026-01-16', 'Direct deposit'),
  (1003, 'Carol White',   4200.00, '2026-01-17', 'Invoice payment')
""")


In [ ]:
spark.sql(f"""
-- New table is fully locked down — all columns masked
SELECT * FROM {catalog}.{schema}.transaction_log
""").show(truncate=False)


---
## Part 2: Data Classification Integration

Instead of manually tagging PII columns, we can use Databricks **Data Classification** to automatically detect sensitive data and apply `class.*` system governed tags. The control tag pattern ensures data remains locked down until a human reviews the auto-classifications.

**Flow:**
1. New table created → inherits `pending` → all columns masked
2. Data Classification scans the table in the background and applies `class.*` tags (e.g., `class.name`, `class.email_address`, `class.us_ssn`)
3. Admin queries `system.data_classification.results` to verify classifications
4. Admin fixes any false positives by unsetting incorrect tags
5. Admin flips control tag to `reviewed` → only `class.*`-tagged columns remain masked

> **Note:** Part 2 builds on Part 1. If running the full notebook top-to-bottom, note that `customer_records` has `abac_pii` tags (not `class.*` tags) — it will not be affected by the new `review_complete_classified` policy. Only `employee_directory` (which receives `class.*` tags in Step 9) will demonstrate the Data Classification integration.

### Prerequisites

Before proceeding, enable Data Classification on your catalog:

1. Go to **Catalog Explorer** > select your catalog > **Details** tab
2. Toggle **Data Classification** on
3. Optionally select specific schemas to include (or leave as all)
4. In the Data Classification results panel, enable **Auto tag** for the classification types you want (e.g., `class.name`, `class.email_address`, `class.us_ssn`)

> Data Classification runs incrementally. New tables are scanned within 24 hours. For this tutorial, we simulate the `class.*` tags manually to avoid waiting for the background scan.

### Step 8: Create a New Table (Locked Down by Default)

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.employee_directory")
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.employee_directory (
  id INT,
  full_name STRING,
  personal_email STRING,
  ssn STRING,
  phone STRING,
  office_location STRING,
  title STRING,
  team STRING
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.employee_directory VALUES
  (1, 'Alice Johnson',  'alice.j@gmail.com',    '123-45-6789', '555-0101', 'NYC HQ Floor 12',    'Staff Engineer',   'Platform'),
  (2, 'Bob Smith',      'bob.smith@yahoo.com',  '234-56-7890', '555-0202', 'LA Office Suite 4',  'Sales Director',   'Enterprise'),
  (3, 'Carol White',    'carol.w@outlook.com',  '345-67-8901', '555-0303', 'Chicago Rm 301',     'Senior Engineer',  'Platform'),
  (4, 'David Lee',      'david.lee@gmail.com',  '456-78-9012', '555-0404', 'Houston Floor 2',    'Marketing Lead',   'Growth'),
  (5, 'Eva Martinez',   'eva.m@hotmail.com',    '567-89-0123', '555-0505', 'Phoenix Building B', 'HR Business Partner', 'People')
""")


In [ ]:
spark.sql(f"""
-- Table inherited pending status — all columns masked
SELECT * FROM {catalog}.{schema}.employee_directory
""").show(truncate=False)


### Step 9: Simulate Data Classification Tags

In production, Data Classification would automatically apply `class.*` tags after scanning. Here we simulate that by applying them manually. These are the same system governed tags that Data Classification would set.

> **Note:** `class.*` tags use an empty-string value. Use `SET TAGS ('class.name' = '')` for safety.

In [ ]:
spark.sql(f"""
-- Simulate Data Classification auto-tagging
ALTER TABLE {catalog}.{schema}.employee_directory
  ALTER COLUMN full_name SET TAGS ('class.name' = '')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.employee_directory
  ALTER COLUMN personal_email SET TAGS ('class.email_address' = '')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.employee_directory
  ALTER COLUMN ssn SET TAGS ('class.us_ssn' = '')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.employee_directory
  ALTER COLUMN phone SET TAGS ('class.phone_number' = '')
""")


### Step 10: Review Classifications

In production, query `system.data_classification.results` to see all detections with confidence scores and sample values. Review for false positives before releasing the data.

In [ ]:
spark.sql(f"""
-- In production, check auto-classification results:
SELECT catalog_name, schema_name, table_name, column_name,
       class_tag, confidence, frequency
FROM system.data_classification.results
WHERE schema_name = '{schema}'
ORDER BY table_name, column_name
""").show(truncate=False)

spark.sql(f"""
-- For this tutorial, verify our simulated tags:
-- SELECT column_name, tag_name, tag_value
-- FROM information_schema.column_tags
-- WHERE schema_name = '{schema}'
--   AND table_name = 'employee_directory'
--   AND tag_name LIKE 'class.%'
-- ORDER BY column_name
""").show(truncate=False)


### Step 11: Handle False Positives (Optional)

If Data Classification incorrectly tagged a column, unset the tag. Manually removed tags will not be reapplied in future scans.

In [ ]:
spark.sql(f"""
-- Example: if office_location was incorrectly tagged as class.location
-- ALTER TABLE {catalog}.{schema}.employee_directory
--   ALTER COLUMN office_location UNSET TAGS ('class.location')
""")


### Step 12: Policy for Reviewed Tables Using `class.*` Tags

Create a policy that masks columns tagged by Data Classification. Because `class.*` tags are key-only, use `has_tag()` with OR to cover multiple classification types.

In [ ]:
spark.sql(f"""
-- New policy: mask columns with any class.* PII tag when table is reviewed
CREATE OR REPLACE POLICY review_complete_classified
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.flexible_mask
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
WHEN has_tag_value('abac_tut_review_status', 'reviewed')
MATCH COLUMNS (
  has_tag('class.name')
  OR has_tag('class.email_address')
  OR has_tag('class.us_ssn')
  OR has_tag('class.phone_number')
  OR has_tag('class.credit_card')
  OR has_tag('class.date_of_birth')
) AS m
ON COLUMN m
""")


### Step 13: Flip Control Tag to Reviewed

After verifying the classifications, mark the table as reviewed. Non-PII columns become visible while classified columns stay masked.

In [ ]:
spark.sql(f"""
ALTER TABLE {catalog}.{schema}.employee_directory
  SET TAGS ('abac_tut_review_status' = 'reviewed')
""")


In [ ]:
spark.sql(f"""
-- id, office_location, title, team are now visible
-- full_name (class.name), personal_email (class.email_address),
-- ssn (class.us_ssn), phone (class.phone_number) remain masked
SELECT * FROM {catalog}.{schema}.employee_directory
""").show(truncate=False)


### Summary: The Full Workflow

| Stage | What happens | Who acts |
|-------|-------------|----------|
| Table created | Inherits `review_status=pending` from schema | Automatic |
| Pending state | ALL columns masked for non-engineers | Policy 1 |
| Classification | Data Classification scans and applies `class.*` tags | Automatic (background) |
| Human review | Engineer queries results, fixes false positives | Admin |
| Flip to reviewed | `ALTER TABLE SET TAGS ('abac_tut_review_status' = 'reviewed')` | Admin |
| Reviewed state | Only `class.*`-tagged columns are masked | Policy 2 |

This pattern gives you **secure-by-default** (nothing visible until reviewed) combined with **automated classification** (no manual PII tagging needed) and **human-in-the-loop** (engineer verifies before releasing).

### Delete governed tags

If you no longer need them, delete `abac_review_status` and `abac_pii` from the Catalog Explorer UI. The `class.*` system governed tags cannot be deleted — they are managed by Databricks.

---
## Industry Examples

> **Instructions:** Replace `{catalog}` with your Unity Catalog catalog name and `{schema}` with your target schema name before running.
>
> Groups referenced in this section (`abac_tut_admins`, `retail_ops`) must be created in the **Account Console** by an admin.
>
> Governed tags `pci_review_status` (with values: pending, reviewed) and `pci_data_type` (with values: card_number, cvv, name) must be created in the Catalog Explorer UI.

## Groups Used in Industry Examples

The industry examples below reuse the same account groups from the core tutorial:

| Group | Used As |
|-------|---------|
| `abac_tut_us_team` | Regional (US/North/Voice) |
| `abac_tut_eu_team` | Regional (EU/South/Data) |
| `abac_tut_apac_team` | Regional (APAC/East/Bundle) |
| `abac_tut_admins` | Admin exemption (all policies) |
| `abac_tut_hr_team` | HR domain / Identity owners |
| `abac_tut_finance_team` | Finance domain / Billing / Fraud analysts |
| `abac_tut_marketing_team` | Marketing domain / CPNI owners |

> These groups are managed by your account admin. No group creation SQL is needed here.

## Retail — PCI-DSS Data Onboarding Review Workflow

When retail systems ingest new transaction data, all fields must be locked down until a PCI compliance officer reviews which columns contain cardholder data. This prevents accidental exposure of card numbers, CVV codes, or billing information during the data ingestion pipeline.

**Workflow:**
1. New table lands in the schema → inherits `pci_review_status=pending` → ALL columns masked
2. PCI compliance officer reviews the data structure
3. Officer tags PCI-specific columns (`pci_data_type` = card_number, cvv, name)
4. Officer flips schema tag to `reviewed` → only tagged columns remain masked
5. Any new table added to the schema automatically enters the review workflow

**Compliance context:** PCI-DSS Requirement 3 mandates protecting stored cardholder data. The secure-by-default pattern ensures no cardholder data is ever accidentally exposed during onboarding.

In [ ]:
spark.sql(f"""
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
-- Set schema-level PCI review status to pending
-- All new tables inherit this tag automatically
ALTER SCHEMA {catalog}.{schema}
  SET TAGS ('pci_review_status' = 'pending')
""")


In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.retail_transactions")
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.retail_transactions (
  txn_id           INT,
  customer_name    STRING,
  card_number      STRING,
  cvv              STRING,
  billing_zip      STRING,
  merchant_name    STRING,
  amount           DOUBLE,
  txn_date         DATE,
  transaction_type STRING
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.retail_transactions VALUES
  (1, 'Alice Brown',  '4532-1234-5678-9010', '123', '10001', 'Amazon',     45.99,   '2026-01-15', 'online'),
  (2, 'Bob Davis',    '5425-2334-3010-9288', '456', '60614', 'Target',     128.50,  '2026-01-16', 'in-store'),
  (3, 'Carol Evans',  '3714-496353-98431',   '789', '78701', 'Apple Store', 999.00, '2026-01-17', 'in-store'),
  (4, 'David Garcia', '4916-3289-1234-5678', '321', '90001', 'Netflix',    15.99,   '2026-01-18', 'subscription'),
  (5, 'Eva Harris',   '5105-1051-0510-5100', '654', '98101', 'Best Buy',   349.00,  '2026-01-19', 'in-store'),
  (6, 'Frank Iyer',   '4111-1111-1111-1111', '987', '10002', 'Whole Foods', 87.25,  '2026-01-20', 'in-store')
""")


### Retail Policy 1: Mask ALL Columns While Pending

A flexible mask UDF that handles STRING columns — the most common type in transaction data. All columns are masked when the table has `pci_review_status=pending`.

In [ ]:
spark.sql(f"""
-- Flexible masking UDF for retail: handles string columns
CREATE OR REPLACE FUNCTION {catalog}.{schema}.retail_flexible_mask(data VARIANT)
RETURNS VARIANT
RETURN CASE
  WHEN schema_of_variant(data) = 'BIGINT' THEN -1::VARIANT
  WHEN schema_of_variant(data) = 'INT'    THEN 0::VARIANT
  WHEN schema_of_variant(data) = 'DOUBLE' THEN 0.00::VARIANT
  WHEN schema_of_variant(data) = 'DATE'   THEN DATE'1970-01-01'::VARIANT
  WHEN schema_of_variant(data) = 'STRING' THEN '"***"'::VARIANT
  WHEN schema_of_variant(data) LIKE 'OBJECT%' THEN parse_json('{{"masked": true}}')
  ELSE NULL::VARIANT
END
""")


In [ ]:
spark.sql(f"""
-- Policy 1: While pci_review_status=pending, mask ALL columns
-- abac_tut_admins are exempt so they can review and tag the data
CREATE OR REPLACE POLICY pci_pending_lockdown
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.retail_flexible_mask
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
WHEN has_tag_value('pci_review_status', 'pending')
MATCH COLUMNS TRUE AS m ON COLUMN m
""")


In [ ]:
spark.sql(f"""
-- Verify lockdown: all columns masked for non-abac_tut_admins
SELECT txn_id, customer_name, card_number, amount, merchant_name
FROM {catalog}.{schema}.retail_transactions
""").show(truncate=False)


### Admin Reviews: Tag PCI Columns

A PCI compliance officer (member of `abac_tut_admins`) reviews the table structure and identifies which columns contain cardholder data per PCI-DSS Requirement 3.

In [ ]:
spark.sql(f"""
-- Admin tags PCI-specific columns after reviewing the table schema
-- These tags identify cardholder data per PCI-DSS Requirement 3
ALTER TABLE {catalog}.{schema}.retail_transactions
  ALTER COLUMN card_number SET TAGS ('pci_data_type' = 'card_number')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.retail_transactions
  ALTER COLUMN cvv SET TAGS ('pci_data_type' = 'cvv')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.retail_transactions
  ALTER COLUMN customer_name SET TAGS ('pci_data_type' = 'name')
""")


### Retail Policy 2: Post-Review — Mask Only PCI-Tagged Columns

Once the PCI officer flips the schema tag to `reviewed`, the pending policy no longer applies. Only PCI-tagged columns (card_number, cvv, customer_name) remain masked. All other columns become visible.

In [ ]:
spark.sql(f"""
-- Credit card number masking: PCI-compliant last-4-digits format
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_card_number(val STRING)
RETURNS STRING
RETURN CONCAT('****-****-****-', RIGHT(REPLACE(val, '-', ''), 4))
""")

spark.sql(f"""
-- CVV must NEVER be stored post-auth per PCI-DSS Req 3.2.2
-- Always fully redact
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_cvv(val STRING)
RETURNS STRING
RETURN '***'
""")

spark.sql(f"""
-- Customer name: partial mask preserving first initial
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_cardholder_name(val STRING)
RETURNS STRING
RETURN CONCAT(LEFT(val, 1), '****')
""")


In [ ]:
spark.sql(f"""
-- Policy 2a: Mask card numbers when reviewed
CREATE OR REPLACE POLICY pci_reviewed_card
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_card_number
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
WHEN has_tag_value('pci_review_status', 'reviewed')
MATCH COLUMNS has_tag_value('pci_data_type', 'card_number') AS m
ON COLUMN m
""")

spark.sql(f"""
-- Policy 2b: Mask CVV when reviewed
CREATE OR REPLACE POLICY pci_reviewed_cvv
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_cvv
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
WHEN has_tag_value('pci_review_status', 'reviewed')
MATCH COLUMNS has_tag_value('pci_data_type', 'cvv') AS m
ON COLUMN m
""")

spark.sql(f"""
-- Policy 2c: Mask cardholder name when reviewed
CREATE OR REPLACE POLICY pci_reviewed_name
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_cardholder_name
TO `account users`
EXCEPT abac_tut_admins
FOR TABLES
WHEN has_tag_value('pci_review_status', 'reviewed')
MATCH COLUMNS has_tag_value('pci_data_type', 'name') AS m
ON COLUMN m
""")


### Flip Review Status — Release the Data

The PCI officer has reviewed and tagged all cardholder data columns. Flip the table to `reviewed` to release non-PCI columns.

In [ ]:
spark.sql(f"""
-- PCI officer approves: flip review status from pending to reviewed
ALTER TABLE {catalog}.{schema}.retail_transactions
  SET TAGS ('pci_review_status' = 'reviewed')
""")


In [ ]:
spark.sql(f"""
-- Non-PCI columns now visible: billing_zip, merchant_name, amount, txn_date, transaction_type
-- PCI columns still masked: customer_name (A****), card_number (****-****-****-XXXX), cvv (***)
SELECT txn_id, customer_name, card_number, cvv, billing_zip, merchant_name, amount, txn_date
FROM {catalog}.{schema}.retail_transactions
""").show(truncate=False)


### New Table Automatically Enters Review Workflow

A new `loyalty_program` table is added to the schema. It automatically inherits `pci_review_status=pending` from the schema tag, locking down all columns until reviewed.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.loyalty_program")
spark.sql(f"""
-- New table added to schema — automatically inherits pending status
CREATE OR REPLACE TABLE {catalog}.{schema}.loyalty_program (
  member_id    INT,
  member_name  STRING,
  email        STRING,
  phone        STRING,
  points       INT,
  tier         STRING,
  joined_date  DATE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.loyalty_program VALUES
  (1, 'Alice Brown',  'alice@example.com',  '617-555-0101', 2500, 'Gold',   '2023-03-15'),
  (2, 'Bob Davis',    'bob@example.com',    '312-555-0202', 800,  'Silver', '2024-01-20'),
  (3, 'Carol Evans',  'carol@example.com',  '512-555-0303', 15000,'Platinum','2022-06-01')
""")


In [ ]:
spark.sql(f"""
-- loyalty_program inherits pci_review_status=pending from schema
-- ALL columns are masked until a PCI officer reviews and tags this table
SELECT * FROM {catalog}.{schema}.loyalty_program
""").show(truncate=False)


### Audit Query: Review Status of All Tables

A PCI compliance dashboard query to see which tables are pending review and which have been cleared.

In [ ]:
spark.sql(f"""
-- Audit: check PCI review status of all tables in the schema
-- Tables without an explicit tag inherit the schema-level pending status
SELECT
  t.table_name,
  COALESCE(tt.tag_value, 'inherited: pending') AS pci_review_status,
  t.created,
  t.last_altered
FROM information_schema.tables t
LEFT JOIN information_schema.table_tags tt
  ON t.table_schema = tt.schema_name
  AND t.table_name = tt.table_name
  AND tt.tag_name = 'pci_review_status'
WHERE t.table_schema = '{schema}'
ORDER BY t.table_name
""").show(truncate=False)
